In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime
import logging        
from config import ROUTES, PipelineConfig  

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# Config 
GOLD_FATO_PATH   = "workspace.case_spark_cvm.gold_fato_diario"
NOME_TABELA      = f"gold_cubo_risco_retorno" 
GOLD_PATH        = f"{ROUTES.TABLE_BASE}.{NOME_TABELA}"
DATA_PROC        = int(datetime.now().strftime("%Y%m%d"))

### 1. gold_fato_diario

In [0]:
gold_fato_diario  = spark.read.table(GOLD_FATO_PATH)

### 2. Encontrando a Ultima Data do Fundo

In [0]:
window_ultima_data = Window.partitionBy("cnpj_fundo_classe")

gold_fato_diario = gold_fato_diario\
    .withColumn("ultima_dt_fundo", f.max(f.col("dt_comptc")).over(window_ultima_data))


### 3. Selecionando as Colunas de Risco e Retorno

In [0]:
gold_cubo_risco_retorno = gold_fato_diario\
    .filter(f.col("dt_comptc") == f.col("ultima_dt_fundo"))\
    .withColumn(
        "classificacao_sharpe",
        f.when(f.col("sharpe_252d") < 0, "RUIM")
         .when((f.col("sharpe_252d") >= 0) & (f.col("sharpe_252d") < 1), "REGULAR")
         .when((f.col("sharpe_252d") >= 1) & (f.col("sharpe_252d") < 2), "BOM")
         .otherwise("EXCELENTE")
    )\
    .select(
        "cnpj_fundo_classe",
        f.col("dt_comptc").alias("dt_referencia"),
        "retorno_252d",
        "volatilidade_252d",
        "vl_patrim_liq",
        "sharpe_252d",
        "sortino_252d",
        "classificacao_sharpe"
    )

### 4. Salvando os dados

In [0]:

log.info(f"Iniciando a escrita da dimensão unificada em: {GOLD_PATH}") 

PipelineConfig.gravar_cubo_gold(
    spark=spark,
    df_cubo=gold_cubo_risco_retorno,
    tabela_destino=GOLD_PATH,
    zorder_cols=["cnpj_fundo_classe", "classificacao_sharpe"],
)


log.info(f"Processamento da {GOLD_PATH} concluído com sucesso!")

In [0]:
display(gold_cubo_risco_retorno)